# Lab 16: Synthetic Data Generation

Even 20-50 inputs for the evaluation dataset is usually not enough. Ideally we need 100 inputs in the dataset. If we have production samples, we can use that as inputs to our dataset. Otherwise we can use synthetic data generation.

Synthetic data generation means using AI to generate data. When doing synthetic data generation, we need to ensure we are having good diversity in the types of inputs that are generated. This follows a three step process:

1. Deciding the dimensions of variation
2. Creating scenario type typles
3. Generating synthetic inputs

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-5")

## Step 1: Decide the dimensions of variation

First step is to decide the dimensions of variation. For example, long input versus short input. That can be one dimension: the length of the input. Complicated language versus simple language can be another dimension. Then our inputs should have:

- Long input with complicated language
- Long input with simple language
- Short input with complicated language
- Short input with simple language

This way we can ensure there is variation in the inputs

So like this, we can get all the different combinations that we want to generate synthetic data for.

For our job description improvement evaluation dataset, we will need the following dimensions:

- Industry (can be technology, marketing, manufacturing, teaching, medicine, or shipping)
- Length (can be less than 100 words, 100-500 words, or more than 500 words)
- Language (can be easy to understand, biased language, confusing language, or jargon-heavy language)
- The type of job (can be on-site, remote, or hybrid)
- Seniorities like freshers, mid-level, or executive

## Step 2: Come up with the scenario combinations

If we try all possible combinations of inputs, we we get almost 500 inputs. This is too much, so a better approach is to write a small python script to randomly choose. Or just ask the LLM to generate some random combinations like below.

In [ ]:
from langchain_core.prompts import PromptTemplate

SCENARIO_TYPE_PROMPT = """
I am designing a Application Tracking System and want to test it with a diverse set of user scenarios. 

Please generate 50 unique combinations (tuples) using the following key dimensions and their possible values: 

- Industry: Technology, Marketing, Manufacturing, Teaching, Medicine, Shipping 
- Length: Less than 100 words, 100 to 500 words, more than 500 words 
- Language: Easy to understand, biased language, confusing, jargon heavy 
- Type: Onsite, Remote, Hybrid 
- Seniority: Fresher, Mid Level, Executive 

Each combination should select one value from each dimension. 

Present the results as a list of tuples, where each tuple contains one value for each dimension in the following order: (Industry, Length, Language, Type, Seniority). 

Ensure that the combinations are varied and realistic.
"""

response = llm.invoke(SCENARIO_TYPE_PROMPT)

print(response.content)

## Step 3: Synthetic generation

Once we have the combinations, we can feed these combinations into an LLM and ask it to generate one input for each combination. By giving specific combinations we increase the chance that the LLM generates good variety of inputs.

In [ ]:
SCENARIO_GENERATION_PROMPT = """
Convert these dimension combinations into realistic job descriptions for an application tracking system. 

Include variations in: 
- Structuring (Free text vs structured with headings) 
- Common typos 
- Natural language patterns 
- Realistic context and urgency 

Include only 1 example per dimension_example. 

<dimension_examples>
(Technology, Less than 100 words, Easy to understand, Onsite, Fresher)
(Marketing, 100 to 500 words, Jargon heavy, Remote, Mid Level)
(Manufacturing, More than 500 words, Confusing, Hybrid, Executive)
(Teaching, Less than 100 words, Biased language, Remote, Fresher)
(Medicine, 100 to 500 words, Easy to understand, Onsite, Executive)
(Shipping, More than 500 words, Jargon heavy, Hybrid, Mid Level)
(Technology, 100 to 500 words, Confusing, Onsite, Mid Level)
(Marketing, Less than 100 words, Biased language, Hybrid, Executive)
(Manufacturing, More than 500 words, Easy to understand, Remote, Fresher)
(Teaching, 100 to 500 words, Jargon heavy, Onsite, Mid Level)
(Medicine, Less than 100 words, Confusing, Hybrid, Executive)
(Shipping, More than 500 words, Biased language, Remote, Fresher)
(Technology, More than 500 words, Easy to understand, Hybrid, Mid Level)
(Marketing, Less than 100 words, Jargon heavy, Onsite, Executive)
(Manufacturing, 100 to 500 words, Biased language, Remote, Mid Level)
(Teaching, More than 500 words, Confusing, Hybrid, Fresher)
(Medicine, Less than 100 words, Easy to understand, Remote, Mid Level)
(Shipping, 100 to 500 words, Jargon heavy, Onsite, Fresher)
(Technology, More than 500 words, Biased language, Remote, Executive)
(Marketing, 100 to 500 words, Confusing, Hybrid, Fresher)
(Manufacturing, Less than 100 words, Easy to understand, Onsite, Mid Level)
(Teaching, More than 500 words, Jargon heavy, Remote, Executive)
(Medicine, 100 to 500 words, Biased language, Hybrid, Fresher)
(Shipping, Less than 100 words, Confusing, Onsite, Mid Level)
(Technology, More than 500 words, Easy to understand, Remote, Fresher)
(Marketing, 100 to 500 words, Jargon heavy, Hybrid, Executive)
(Manufacturing, Less than 100 words, Biased language, Remote, Mid Level)
(Teaching, More than 500 words, Confusing, Onsite, Fresher)
(Medicine, 100 to 500 words, Easy to understand, Hybrid, Executive)
(Shipping, Less than 100 words, Jargon heavy, Remote, Mid Level)
(Technology, Less than 100 words, Biased language, Hybrid, Executive)
(Marketing, More than 500 words, Confusing, Onsite, Mid Level)
(Manufacturing, 100 to 500 words, Easy to understand, Remote, Fresher)
(Teaching, Less than 100 words, Jargon heavy, Hybrid, Executive)
(Medicine, More than 500 words, Confusing, Onsite, Mid Level)
(Shipping, 100 to 500 words, Biased language, Remote, Fresher)
(Technology, More than 500 words, Easy to understand, Hybrid, Executive)
(Marketing, Less than 100 words, Confusing, Remote, Mid Level)
(Manufacturing, More than 500 words, Jargon heavy, Onsite, Fresher)
(Teaching, 100 to 500 words, Biased language, Hybrid, Executive)
(Medicine, Less than 100 words, Confusing, Remote, Fresher)
(Shipping, More than 500 words, Easy to understand, Onsite, Mid Level)
(Technology, 100 to 500 words, Jargon heavy, Hybrid, Fresher)
(Marketing, Less than 100 words, Biased language, Onsite, Executive)
(Manufacturing, More than 500 words, Confusing, Remote, Mid Level)
(Teaching, 100 to 500 words, Easy to understand, Hybrid, Fresher)
(Medicine, More than 500 words, Jargon heavy, Onsite, Executive)
(Shipping, Less than 100 words, Confusing, Remote, Mid Level)
(Technology, More than 500 words, Biased language, Hybrid, Fresher)
(Marketing, 100 to 500 words, Easy to understand, Onsite, Executive)
</dimension_examples> 

Return the results in json format as a list of strings.
"""

response = llm.invoke(SCENARIO_GENERATION_PROMPT)

print(response.content)

## Step 4: Review

Note that LLMs may not follow the instructions exactly. We will still get some duplicated inputs and inputs that do not follow the combination. So we must manually review the dataset and remove duplicates and poor quality inputs. After that we can add all these inputs into the evaluation dataset